### Average Word2Vec
- This uses the Google's pre-trained word2vec model but we can also use our own custom model.

### gensim.utils.simple_preprocess() function
- simple_preprocess() is a utility function from Gensim (gensim.utils.simple_preprocess). It is used for basic text normalization and tokenization.
- It does the following:-
    - Function behavior (by default)
    - simple_preprocess(text) performs the following steps:
    - Lowercases the text
    - Removes punctuation and special characters
    - Removes numbers
    - Tokenizes the text into individual words
    - Filters out very short or very long tokens
    - Default: keeps tokens with length between 2 and 15 characters
    - Returns a list of tokens (words)


In [8]:
import gensim
from gensim.models import Word2Vec, keyedvectors
import gensim.downloader as api
import pandas as pd
import re # Regular Expressions
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from nltk import sent_tokenize
from gensim.utils import simple_preprocess
from tqdm import tqdm


[nltk_data] Downloading package stopwords to /Users/ankur/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# Step -1) Load the Google News pre-trained Word2Vec model
# word2vec-google-news-300 = Name of Google's pre-trained Word2Vec model.
# This is a 1600 MB download and may take a few minutes to complete.
google_pretrained_model = api.load("word2vec-google-news-300")

In [4]:
# Step-2) Load the Data from the  the CSV file
# - \t = Key value pairs in the CSV file is seperated by Tab
# - label = Key in the CSV
# - message = Value in the CSV
unprocessed_corpus = pd.read_csv('./resources/SMSSpamCollection.txt',sep='\t',names=["label","message"])
print('documents =',unprocessed_corpus)

documents =      label                                            message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
...    ...                                                ...
5567  spam  This is the 2nd time we have tried 2 contact u...
5568   ham               Will ü b going to esplanade fr home?
5569   ham  Pity, * was in mood for that. So...any other s...
5570   ham  The guy did some bitching but I acted like i'd...
5571   ham                         Rofl. Its true to its name

[5572 rows x 2 columns]


In [5]:
# Step-3) Feature Engineering - Data Cleaning and Pre-Processing

# Step 3.1) Text Pre-Processing and Normalization
# - Break the 'unprocessed_corpus' into sentences/documents
# - Remove Special Characters from each sentence/document
# - Convert each sentence into lowercase
# - Remove Stop words from each sentence/document
# - Apply Lemmatization using WordNetLemmatizer on each sentence/document
# - Rejoin all the sentences/documents into 'text_preprocessed_corpus'

word_lemmatizer = WordNetLemmatizer()
text_preprocessed_corpus=[]
for i in range(0,len(unprocessed_corpus)):
    document=re.sub('[^a-zA-Z]',' ',unprocessed_corpus['message'][i]) # Replace all special characters (i.e. Not (a-z or A-Z) characters with Space)
    document=document.lower() # Convert each sentence/document to lowercase
    document=document.split() # Split the sentence/document into individual words, this will create an array of words in each sentence/words
    document = [word_lemmatizer.lemmatize(word) for word in document if not word in stopwords.words('english')] # Remove stop words first, then apply stemming on the remaing sentence/document
    document = ' '.join(document) # Rejoin indidvidual words back into a sentence/document
    text_preprocessed_corpus.append(document) # Rejoin individual sentences/document into 'text_preprocessed_corpus'
print('text_preprocessed_corpus = ',text_preprocessed_corpus)

text_preprocessed_corpus =  ['go jurong point crazy available bugis n great world la e buffet cine got amore wat', 'ok lar joking wif u oni', 'free entry wkly comp win fa cup final tkts st may text fa receive entry question std txt rate c apply', 'u dun say early hor u c already say', 'nah think go usf life around though', 'freemsg hey darling week word back like fun still tb ok xxx std chgs send rcv', 'even brother like speak treat like aid patent', 'per request melle melle oru minnaminunginte nurungu vettam set callertune caller press copy friend callertune', 'winner valued network customer selected receivea prize reward claim call claim code kl valid hour', 'mobile month u r entitled update latest colour mobile camera free call mobile update co free', 'gonna home soon want talk stuff anymore tonight k cried enough today', 'six chance win cash pound txt csh send cost p day day tsandcs apply reply hl info', 'urgent week free membership prize jackpot txt word claim c www dbuk net lcclt

In [6]:
# Step-3.2) Feature Engineering - Further Text Pre-Processing and Normalization  - using gensim.utils.simple_preprocess() function
#- simple_preprocess() is a utility function from Gensim (gensim.utils.simple_preprocess). It is used for basic text normalization and tokenization.
#- It does the following:-
#   - Function behavior (by default)
#    - simple_preprocess(text) performs the following steps:
#    - Lowercases the text
#    - Removes punctuation and special characters
#    - Removes numbers
#    - Tokenizes the text into individual words
#    - Filters out very short or very long tokens
#    - Default: keeps tokens with length between 2 and 15 characters
#    - Returns a list of tokens (words)

text_processed_words=[]
for sentence in text_preprocessed_corpus: # Break the 'text_preprocessed_corpus' into sentences/documents
    words_in_sentence = sent_tokenize(sentence) # Break each sentence/document into words
    for word in words_in_sentence: # Perform normalization and text pre-processing on each word
        text_processed_words.append(simple_preprocess(word))
print('text_processed_words = ',text_processed_words)


print('sentence[0] = ',text_processed_words[0])
    

text_processed_words =  [['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'great', 'world', 'la', 'buffet', 'cine', 'got', 'amore', 'wat'], ['ok', 'lar', 'joking', 'wif', 'oni'], ['free', 'entry', 'wkly', 'comp', 'win', 'fa', 'cup', 'final', 'tkts', 'st', 'may', 'text', 'fa', 'receive', 'entry', 'question', 'std', 'txt', 'rate', 'apply'], ['dun', 'say', 'early', 'hor', 'already', 'say'], ['nah', 'think', 'go', 'usf', 'life', 'around', 'though'], ['freemsg', 'hey', 'darling', 'week', 'word', 'back', 'like', 'fun', 'still', 'tb', 'ok', 'xxx', 'std', 'chgs', 'send', 'rcv'], ['even', 'brother', 'like', 'speak', 'treat', 'like', 'aid', 'patent'], ['per', 'request', 'melle', 'melle', 'oru', 'minnaminunginte', 'nurungu', 'vettam', 'set', 'callertune', 'caller', 'press', 'copy', 'friend', 'callertune'], ['winner', 'valued', 'network', 'customer', 'selected', 'receivea', 'prize', 'reward', 'claim', 'call', 'claim', 'code', 'kl', 'valid', 'hour'], ['mobile', 'month', 'entitled', 'update'

In [ ]:
# Step-4) Train Google's model on our Vocabulary (i.e. text_processed_words)
google_pretrained_model = gensim.models.Word2Vec(text_processed_words)


In [24]:
# Step-5)  Use pre-defined word functions provied by this model.

# Print the vocabulary learnt by Google's pre-trained Word2Vec model
# wv = Word Vectors
# index_to_key = List of words in the vocabulary learnt by this model
print('Vocabulary: ',google_pretrained_model.wv.index_to_key)

# Total number of words in the vocabulary learnt by this model
print('Corpus Count: ',google_pretrained_model.corpus_count)

# model.epochs = Number of iterations (epochs) the model has been trained on
# Higher this number, more accurate results we will get, and more time it will take.
print('Epochs: ',google_pretrained_model.epochs)

# Print similar words to the one provided (here 'kid')
print('words similar to kid: ',google_pretrained_model.wv.similar_by_word('kid'))

# Get vector representation of a word (='good' in this case)
google_pretrained_model.wv['good']  # Get vector representation of a word (='good' in this case)
print(f"Vector representation of word 'good' is : \n{google_pretrained_model.wv['good']}")

google_pretrained_model.wv['good'].shape  # Shape of this vector is 100 dimensions
print('vector_good.shape = ',google_pretrained_model.wv['good'].shape)


Vocabulary:  ['call', 'get', 'ur', 'gt', 'lt', 'go', 'day', 'ok', 'free', 'know', 'come', 'like', 'good', 'time', 'got', 'love', 'text', 'want', 'send', 'one', 'need', 'txt', 'today', 'going', 'stop', 'home', 'lor', 'sorry', 'see', 'still', 'mobile', 'take', 'back', 'da', 'reply', 'dont', 'think', 'tell', 'week', 'phone', 'hi', 'new', 'later', 'please', 'pls', 'co', 'msg', 'min', 'dear', 'night', 'make', 'message', 'well', 'say', 'thing', 'much', 'hope', 'oh', 'claim', 'great', 'hey', 'number', 'give', 'happy', 'work', 'friend', 'wat', 'way', 'yes', 'www', 'let', 'prize', 'right', 'tomorrow', 'already', 'tone', 'said', 'ask', 'win', 'amp', 'cash', 'life', 'im', 'yeah', 'really', 'babe', 'meet', 'find', 'miss', 'morning', 'last', 'service', 'year', 'uk', 'thanks', 'care', 'would', 'anything', 'com', 'also', 'nokia', 'lol', 'every', 'feel', 'keep', 'pick', 'sure', 'sent', 'contact', 'urgent', 'something', 'gud', 'buy', 'cant', 'wait', 'box', 'place', 'first', 'even', 'guy', 'someone', 'h

In [20]:
# Step-6) Average Word2Vec
# - text_processed_words[] (from Step 3.2) is an array which contains all sentences and words
# - text_processed_words[0] = sentence 1
# - text_processed_words[2] = sentence 2 ...
# If we use 'text_processed_words' to generate the Vectors, it will generate Vectors for each word in each sentence.
# But we need a single Vector for each sentence/document.
# So, we will calculate the Average of all word Vectors in each sentence/document to get a single Vector for that sentence/document.

# Create a function to compute the average word2vec for a document
def avg_word2vec(document):
    return np.mean([google_pretrained_model[word] for word in document if word in google_pretrained_model.index_to_key])

# Apply average word2vec to all documents in the corpus using the function 'avg_word2vec()' defined above.
# tqdm is to add a smart progress meter to loops, making it easy to track how long tasks are taking and how much work is left.
x_avg_word2vec_vector=[]
for i in tqdm(range(len(text_processed_words))):
    x_avg_word2vec_vector.append(avg_word2vec(text_processed_words[i]))

print('x_avg_word2vec_vector=',x_avg_word2vec_vector)
print('x_avg_word2vec_vector dimensions=',len(x_avg_word2vec_vector))


100%|██████████| 5564/5564 [02:05<00:00, 44.23it/s]

x_avg_word2vec_vector= [np.float32(-0.0012394808), np.float32(-0.006494351), np.float32(-0.006726059), np.float32(-0.004572293), np.float32(-0.0033712692), np.float32(-6.093514e-05), np.float32(-0.0056560063), np.float32(0.0010608635), np.float32(-0.0072688838), np.float32(-0.003762547), np.float32(-0.00013166096), np.float32(-0.0037394841), np.float32(0.0017954682), np.float32(-0.003140718), np.float32(0.0024246252), np.float32(-0.006214933), np.float32(0.007441271), np.float32(-0.00029325922), np.float32(-0.0006634267), np.float32(-0.005132841), np.float32(-0.008307084), np.float32(0.0013755857), np.float32(-0.004641918), np.float32(-0.006853745), np.float32(0.0020129932), np.float32(-0.0049746707), np.float32(-0.006242955), np.float32(-0.0067746947), np.float32(-0.004155762), np.float32(-0.0021151474), np.float32(-0.0024334907), np.float32(-0.0020067547), np.float32(-0.0067855516), np.float32(-0.0029241466), np.float32(-0.00078822137), np.float32(0.00130313), np.float32(0.0002191696

In [25]:
# Step 7) Get the Input(x_avg_word2vec_vector_array) and Output(y) features for Model Training

# Input feature: x_avg_word2vec_vector_array
# Convert x_avg_word2vec_vector to numpy array
x_avg_word2vec_vector_array = np.array(x_avg_word2vec_vector)
print('x_avg_word2vec_vector_array shape = ',x_avg_word2vec_vector_array.shape)  # Shape of x_avg_word2vec_vector_array will be (5572, 300)

# Dimensions of a vector for a single sentence/document
print('Dimensions of a vector for a single sentence/document  = ',x_avg_word2vec_vector_array[0].shape)

# Output feature: y
y = pd.get_dummies(unprocessed_corpus['label'])
y=y.iloc[:,0].values  # Convert the 'y' dataframe to array
print('y = ',y)

x_avg_word2vec_vector_array shape =  (5564,)
Dimensions of a vector for a single sentence/document  =  ()
y =  [ True  True False ...  True  True  True]
